# Phase 1 - Data Ingestion, Audit and Cleaning

This notebook loads the three Moodwave datasets, checks their structure, cleans them, and saves clean files for the next phase.

In [1]:
import os
import json
import pandas as pd
import numpy as np

# project starts from /notebook, so we gotta go back to the project root /moodwave/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working folder:", os.getcwd())

Working folder: C:\dev\Moodwave\moodwave-ml


## 1. Load the historical playlist data

In [2]:
historical_raw = pd.read_csv(
    "data/raw/playlist_2010to2023.csv",
    encoding="cp1252"
)

historical_raw.head()

,playlist_url,year,track_id,track_name,track_popularity,album,artist_id,artist_name,artist_genres,artist_popularity,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,6naxalmIoLFWR0siv8dnQQ,Oops!...I Did It Again,81,Oops!... I Did It Again,26dSoYclwsYLMAKD3tpOr4,Britney Spears,"['dance pop', 'pop']",81,...,-5.444,0,0.0437,0.3000,0.000018,0.3550,0.894,95.053,211160,4
1,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,2m1hi0nfMR9vdGC8UcrnwU,All The Small Things,83,Enema Of The State,6FBDaR13swtiWwGhX1WQsP,blink-182,"['alternative metal', 'modern rock', 'pop punk...",79,...,-4.918,1,0.0488,0.0103,0.000000,0.6120,0.684,148.726,167067,4
2,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,3y4LxiYMgDl4RethdzpmNe,Breathe,66,Breathe,25NQNriVT2YbSW80ILRWJa,Faith Hill,"['contemporary country', 'country', 'country d...",62,...,-9.007,1,0.0290,0.1730,0.000000,0.2510,0.278,136.859,250547,4
3,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,0v1XpBHnsbkCn7iJ9Ucr1l,It's My Life,81,Crush,58lV9VcRSjABbAbfWS6skp,Bon Jovi,"['glam metal', 'rock']",79,...,-4.063,0,0.0466,0.0263,0.000013,0.3470,0.544,119.992,224493,4
4,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,62bOmKYxYg7dhrC6gH9vFn,Bye Bye Bye,75,No Strings Attached,6Ff53KvcvAj5U7Z1vojB5o,*NSYNC,"['boy band', 'dance pop', 'pop']",70,...,-4.843,0,0.0479,0.0310,0.001200,0.0821,0.861,172.638,200400,4


In [6]:
print("Shape:", historical_raw.shape) # no of rows and columns displayed
historical_raw.info() # information about the dataset such as - datatype, non-null counts

Shape: (2400, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   playlist_url       2400 non-null   object 
 1   year               2400 non-null   int64  
 2   track_id           2400 non-null   object 
 3   track_name         2400 non-null   object 
 4   track_popularity   2400 non-null   int64  
 5   album              2400 non-null   object 
 6   artist_id          2400 non-null   object 
 7   artist_name        2400 non-null   object 
 8   artist_genres      2400 non-null   object 
 9   artist_popularity  2400 non-null   int64  
 10  danceability       2400 non-null   float64
 11  energy             2400 non-null   float64
 12  key                2400 non-null   int64  
 13  loudness           2400 non-null   float64
 14  mode               2400 non-null   int64  
 15  speechiness        2400 non-null   float64
 16  acoust

In [7]:
print("Missing values:")
print(historical_raw.isnull().sum()) # total number of null/missing valuess

# reading col year for seeing kun year dekhi kun year samma xa
print("\nYears:", historical_raw['year'].min(), "to", historical_raw['year'].max())

# total unique tracks herna
print("Unique tracks:", historical_raw['track_id'].nunique())

Missing values:
playlist_url         0
year                 0
track_id             0
track_name           0
track_popularity     0
album                0
artist_id            0
artist_name          0
artist_genres        0
artist_popularity    0
danceability         0
energy               0
key                  0
loudness             0
mode                 0
speechiness          0
acousticness         0
instrumentalness     0
liveness             0
valence              0
tempo                0
duration_ms          0
time_signature       0
dtype: int64

Years: 2000 to 2023
Unique tracks: 2302


## 2. Load the genre data

The source file has two known formatting problems: many rows have two empty values at the end, and one song title contains unquoted commas. We repair only these known cases instead of skipping the rows.

In [4]:
repair_count = {"trailing_empty_values": 0, "split_title": 0, "quarantined": 0}
quarantined_rows = []

# Pandas calls this only when a row has the wrong number of CSV values.
def fix_genre_row(row):
    # Most bad rows only contain two extra empty values at the end.
    if len(row) == 23 and row[-2:] == ["", ""]:
        repair_count["trailing_empty_values"] += 1
        return row[:-2]

    # One title contains commas that were not quoted correctly in the CSV.
    if len(row) == 23:
        fixed = row[:4] + [",".join(row[4:7]).strip('"')] + row[7:]
        if len(fixed) == 21:
            repair_count["split_title"] += 1
            return fixed

    repair_count["quarantined"] += 1
    quarantined_rows.append(row)
    return None

# loading the genre dataset
genre_raw = pd.read_csv(
    "data/raw/genre_specific_dataset.csv",

    # calling fix_genre_row for row/lines 
    engine="python",
    on_bad_lines=fix_genre_row
)

genre_raw.head()

,SN,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
print("Shape:", genre_raw.shape)
print("Repairs:", repair_count)
print("Genres:", genre_raw['track_genre'].nunique())

genre_raw.info()

NameError: name 'genre_raw' is not defined

In [10]:
print("Missing values:")
print(genre_raw.isnull().sum())

Missing values:
SN                  0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64


## 3. Load the country chart data

The full file is large, so it is read in chunks. The supplied development file is small, but the same code also works when it is replaced by the full dataset.

In [11]:
country_chunks = []



for chunk in pd.read_csv(
    "data/raw/top_song_by_countries.csv",
    chunksize=100000
):
    country_chunks.append(chunk)


country_raw = pd.concat(country_chunks, ignore_index=True)
country_raw.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,NaN,2025-06-11,95,False,...,2,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,NaN,2025-06-11,89,True,...,7,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,NaN,2025-06-11,98,False,...,1,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4
3,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,"Lady Gaga, Bruno Mars",4,0,-1,NaN,2025-06-11,91,False,...,6,-7.727,0,0.0317,0.289000,0.000000,0.1260,0.498,157.964,3
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,0,NaN,2025-06-11,100,False,...,2,-10.171,1,0.0358,0.200000,0.060800,0.1170,0.438,104.978,4


In [12]:
print("Shape:", country_raw.shape)
print("Missing values:")
print(country_raw.isnull().sum())

Shape: (2110316, 25)
Missing values:
spotify_id                0
name                     30
artists                  29
daily_rank                0
daily_movement            0
weekly_movement           0
country               28908
snapshot_date             0
popularity                0
is_explicit               0
duration_ms               0
album_name              822
album_release_date      659
danceability              0
energy                    0
key                       0
loudness                  0
mode                      0
speechiness               0
acousticness              0
instrumentalness          0
liveness                  0
valence                   0
tempo                     0
time_signature            0
dtype: int64


## 4. Clean the historical data

In [13]:
historical = historical_raw.copy()

# Use the same names for common columns across all three datasets.
historical = historical.rename(columns={
    "track_popularity": "popularity",
    "album": "album_name"
})

# Remove spaces around text values.
text_columns = historical.select_dtypes(include="object").columns
for column in text_columns:
    historical[column] = historical[column].str.strip()

# A track can appear in different years, so track_id + year is the duplicate key.
historical_duplicates = historical.duplicated(
    subset=["track_id", "year"]
).sum()

historical = historical.drop_duplicates(
    subset=["track_id", "year"]
).reset_index(drop=True)

print("Duplicate historical rows removed:", historical_duplicates)
print("Clean shape:", historical.shape)

Duplicate historical rows removed: 0
Clean shape: (2400, 23)


## 5. Clean the genre data

In [14]:
genre = genre_raw.copy()

genre = genre.rename(columns={
    "SN": "source_row_id",
    "artists": "artist_name"
})

# remove trailing whitespaces
text_columns = genre.select_dtypes(include="object").columns
for column in text_columns:
    genre[column] = genre[column].str.strip()

# The same track may have different genres, so track_id + track_genre is the duplicate key.
# removing duplicated tracks
genre_duplicates = genre.duplicated(
    subset=["track_id", "track_genre"]
).sum()

genre = genre.drop_duplicates(
    subset=["track_id", "track_genre"]
).reset_index(drop=True)

print("Duplicate genre rows removed:", genre_duplicates)
print("Clean shape:", genre.shape)

Duplicate genre rows removed: 450
Clean shape: (113550, 21)


## 6. Clean the country chart data

In [15]:
country = country_raw.copy()

country = country.rename(columns={
    "spotify_id": "track_id",
    "name": "track_name",
    "artists": "artist_name",
    "is_explicit": "explicit"
})

# Blank country means the World Top / global chart.
country['country'] = country['country'].fillna('GLOBAL')
country['country'] = country['country'].astype(str).str.strip().str.upper()

# Convert date columns to real datetime values.
country['snapshot_date'] = pd.to_datetime(
    country['snapshot_date'], errors='coerce'
)
country['album_release_date'] = pd.to_datetime(
    country['album_release_date'], errors='coerce'
)

text_columns = country.select_dtypes(include="object").columns
for column in text_columns:
    country[column] = country[column].str.strip()

# The same track can chart on many dates and in many countries.
country_duplicates = country.duplicated(
    subset=["track_id", "country", "snapshot_date", "daily_rank"]
).sum()

country = country.drop_duplicates(
    subset=["track_id", "country", "snapshot_date", "daily_rank"]
).reset_index(drop=True)

print("Duplicate country-chart rows removed:", country_duplicates)
print("Clean shape:", country.shape)
print("Chart scopes:")
print(country['country'].value_counts().head(20))

Duplicate country-chart rows removed: 0
Clean shape: (2110316, 25)
Chart scopes:
country
DO    29176
IT    29174
NI    29170
PL    29164
HU    29163
SV    29162
HN    29162
CR    29161
TH    29161
EG    29161
CZ    29161
KZ    29161
FI    29160
FR    29159
PT    29159
PK    29159
PA    29158
GT    29158
BR    29157
MY    29157
Name: count, dtype: int64


## 7. Check important numeric ranges

In [16]:
# These Spotify features should normally stay between 0 and 1.
zero_to_one = [
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence'
]

range_rows = []

for dataset_name, df in {
    'historical': historical,
    'genre': genre,
    'country': country
}.items():
    for column in zero_to_one:
        invalid = ((df[column] < 0) | (df[column] > 1)).sum()
        range_rows.append([dataset_name, column, int(invalid)])

    range_rows.append([
        dataset_name,
        'popularity',
        int(((df['popularity'] < 0) | (df['popularity'] > 100)).sum())
    ])

    range_rows.append([
        dataset_name,
        'key',
        int(((df['key'] < 0) | (df['key'] > 11)).sum())
    ])

range_check = pd.DataFrame(
    range_rows,
    columns=['dataset', 'column', 'invalid_values']
)

range_check[range_check['invalid_values'] > 0]

,dataset,column,invalid_values


## 8. Create the data-quality report

In [17]:
# for csv quality report
quality_summary = pd.DataFrame({
    'dataset': ['historical', 'genre', 'country'],

    'raw_rows': [len(historical_raw), len(genre_raw), len(country_raw)],

    'clean_rows': [len(historical), len(genre), len(country)],

    'duplicates_removed': [
        int(historical_duplicates),
        int(genre_duplicates),
        int(country_duplicates)
    ],
    'unique_tracks': [
        historical['track_id'].nunique(),
        genre['track_id'].nunique(),
        country['track_id'].nunique()
    ],

    'missing_values_after_cleaning': [
        int(historical.isnull().sum().sum()),
        int(genre.isnull().sum().sum()),
        int(country.isnull().sum().sum())
    ]
})

quality_summary

,dataset,raw_rows,clean_rows,duplicates_removed,unique_tracks,missing_values_after_cleaning
0,historical,2400,2400,0,2302,0
1,genre,114000,113550,450,89741,3
2,country,2110316,2110316,0,24983,1540


In [18]:
# Save a simple JSON summary as well as the CSV report.
quality_json = {
    'historical': {
        'raw_rows': int(len(historical_raw)),
        'clean_rows': int(len(historical)),
        'year_min': int(historical['year'].min()),
        'year_max': int(historical['year'].max()),
        'duplicates_removed': int(historical_duplicates)
    },
    'genre': {
        'raw_rows': int(len(genre_raw)),
        'clean_rows': int(len(genre)),
        'genre_count': int(genre['track_genre'].nunique()),
        'duplicates_removed': int(genre_duplicates),
        'repairs': repair_count
    },
    'country': {
        'raw_rows': int(len(country_raw)),
        'clean_rows': int(len(country)),
        'global_rows': int((country['country'] == 'GLOBAL').sum()),
        'duplicates_removed': int(country_duplicates)
    }
}

os.makedirs('reports/data_quality', exist_ok=True)
quality_summary.to_csv(
    'reports/data_quality/data_quality_summary.csv', index=False
)
range_check.to_csv(
    'reports/data_quality/range_validation.csv', index=False
)

pd.DataFrame(
    quarantined_rows,
    columns=genre_raw.columns if len(quarantined_rows) > 0 else ['raw_record']
).to_csv('reports/data_quality/quarantined_rows.csv', index=False)

with open('reports/data_quality/data_quality_summary.json', 'w') as f:
    json.dump(quality_json, f, indent=4)

## 9. Save the clean Phase 1 datasets

In [19]:
os.makedirs('data/interim', exist_ok=True)

# Compressed CSV copies are easy to inspect manually.
historical.to_csv(
    'data/interim/historical_tracks_clean.csv.gz',
    index=False,
    compression='gzip'
)
genre.to_csv(
    'data/interim/genre_tracks_clean.csv.gz',
    index=False,
    compression='gzip'
)
country.to_csv(
    'data/interim/country_chart_clean.csv.gz',
    index=False,
    compression='gzip'
)

# Parquet is the main format used by later phases when pyarrow is available.
try:
    historical.to_parquet('data/interim/historical_tracks_clean.parquet', index=False)
    genre.to_parquet('data/interim/genre_tracks_clean.parquet', index=False)
    country.to_parquet('data/interim/country_chart_clean.parquet', index=False)
    print("Parquet files saved.")
except ImportError:
    print("PyArrow is not installed, so only the CSV copies were saved.")

print("Phase 1 complete.")

Parquet files saved.
Phase 1 complete.




- Raw files were left unchanged.
- Historical coverage is kept as **2000-2023**.
- Known malformed genre rows were repaired instead of skipped.
- Blank country values were converted to **GLOBAL**, meaning World Top.
- Repeated songs were only removed when they duplicated the correct dataset-specific key.
- Native Spotify values were kept; no scaling was applied in this phase.